<a href="https://colab.research.google.com/github/saim9211/Machine-Learning-Stuff/blob/main/Elastic_net_Regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("krishd123/high-dimensional-datascape")

print("Path to dataset files:", path)

100%|██████████| 389k/389k [00:00<00:00, 55.7MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/krishd123/high-dimensional-datascape/versions/1


In [3]:
import os
import pandas as pd

# List files in the downloaded dataset path
print(f"Files in dataset directory '{path}':")
for root, dirs, files in os.walk(path):
    for file in files:
        print(os.path.join(root, file))

# Assuming there is a CSV file directly in the path or a subdirectory
# Let's try to find a common data file (e.g., CSV)
dataset_file = None
for root, dirs, files in os.walk(path):
    for file in files:
        if file.endswith('.csv'): # Or other common data formats like .json, .parquet
            dataset_file = os.path.join(root, file)
            break
    if dataset_file:
        break

if dataset_file:
    print(f"\nLoading {dataset_file} into a DataFrame...")
    df = pd.read_csv(dataset_file)
    print("DataFrame loaded successfully!")
    print("First 5 rows:")
    print(df.head())
    print("\nDataFrame info:")
    df.info()
else:
    print("No suitable data file (e.g., .csv) found in the dataset directory.")


Files in dataset directory '/root/.cache/kagglehub/datasets/krishd123/high-dimensional-datascape/versions/1':
/root/.cache/kagglehub/datasets/krishd123/high-dimensional-datascape/versions/1/all_data.csv

Loading /root/.cache/kagglehub/datasets/krishd123/high-dimensional-datascape/versions/1/all_data.csv into a DataFrame...
DataFrame loaded successfully!
First 5 rows:
   Unnamed: 0  Unnamed: 1  Unnamed: 2  Unnamed: 3  Unnamed: 4  Unnamed: 5  \
0   -0.000133    0.000262    0.001099    0.001834    0.002109    0.002223   
1   -0.000842   -0.001011   -0.001071   -0.000944   -0.000794   -0.000610   
2   -0.000766   -0.000535    0.000162    0.000898    0.001287    0.001582   
3   -0.000301   -0.000377   -0.000451   -0.000529   -0.000685   -0.000845   
4   -0.000589   -0.000857   -0.001135   -0.001171   -0.001128   -0.001039   

   Unnamed: 6  Unnamed: 7  Unnamed: 8  Unnamed: 9  ...  Unnamed: 527  \
0    0.002233    0.002036    0.001582    0.000969  ...       0.82953   
1   -0.000445   -0.0001

In [4]:
import numpy as np
import pandas as pd
df.sample(2)

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 527,Unnamed: 528,Unnamed: 529,Unnamed: 530,Unnamed: 531,Unnamed: 532,Unnamed: 533,Unnamed: 534,Unnamed: 535,Label
105,-0.001011,-0.001539,-0.002164,-0.002276,-0.002072,-0.001809,-0.001558,-0.001338,-0.001099,-0.000708,...,0.79522,2.6454,3.1160,1.1842,1.32380,0.53989,0.75575,2.2581,1.87040,0
1,-0.000842,-0.001011,-0.001071,-0.000944,-0.000794,-0.000610,-0.000445,-0.000173,0.000077,0.000285,...,0.84335,3.0110,3.9877,1.2461,0.74423,0.22567,0.61034,1.6645,0.74574,0


In [6]:
import re

# Rename 'Unnamed: X' columns to 'feature_X'
def rename_columns(df):
    new_columns = {}
    for col in df.columns:
        match = re.match(r'Unnamed: (\d+)', col)
        if match:
            new_columns[col] = f"feature_{match.group(1)}"
        else:
            new_columns[col] = col # Keep other columns (like 'Label') as is
    return df.rename(columns=new_columns)

df = rename_columns(df)

print("Columns after renaming:")
print(df.columns.tolist())

print("\nFirst 5 rows with new column names:")
display(df.head())

Columns after renaming:
['feature_0', 'feature_1', 'feature_2', 'feature_3', 'feature_4', 'feature_5', 'feature_6', 'feature_7', 'feature_8', 'feature_9', 'feature_10', 'feature_11', 'feature_12', 'feature_13', 'feature_14', 'feature_15', 'feature_16', 'feature_17', 'feature_18', 'feature_19', 'feature_20', 'feature_21', 'feature_22', 'feature_23', 'feature_24', 'feature_25', 'feature_26', 'feature_27', 'feature_28', 'feature_29', 'feature_30', 'feature_31', 'feature_32', 'feature_33', 'feature_34', 'feature_35', 'feature_36', 'feature_37', 'feature_38', 'feature_39', 'feature_40', 'feature_41', 'feature_42', 'feature_43', 'feature_44', 'feature_45', 'feature_46', 'feature_47', 'feature_48', 'feature_49', 'feature_50', 'feature_51', 'feature_52', 'feature_53', 'feature_54', 'feature_55', 'feature_56', 'feature_57', 'feature_58', 'feature_59', 'feature_60', 'feature_61', 'feature_62', 'feature_63', 'feature_64', 'feature_65', 'feature_66', 'feature_67', 'feature_68', 'feature_69', 'feat

,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,...,feature_527,feature_528,feature_529,feature_530,feature_531,feature_532,feature_533,feature_534,feature_535,Label
0,-0.000133,0.000262,0.001099,0.001834,0.002109,0.002223,0.002233,0.002036,0.001582,0.000969,...,0.82953,2.9079,3.7557,1.3344,0.74247,0.22507,0.56249,1.5705,0.79906,0
1,-0.000842,-0.001011,-0.001071,-0.000944,-0.000794,-0.000610,-0.000445,-0.000173,0.000077,0.000285,...,0.84335,3.0110,3.9877,1.2461,0.74423,0.22567,0.61034,1.6645,0.74574,0
2,-0.000766,-0.000535,0.000162,0.000898,0.001287,0.001582,0.001704,0.001659,0.001574,0.001438,...,0.87413,3.0613,3.9749,1.1560,0.52508,0.19934,0.45707,1.3386,0.74574,0
3,-0.000301,-0.000377,-0.000451,-0.000529,-0.000685,-0.000845,-0.000899,-0.000822,-0.000550,-0.000182,...,0.85467,3.3337,3.9205,1.3341,0.46024,0.20031,0.45924,1.7969,0.32451,0
4,-0.000589,-0.000857,-0.001135,-0.001171,-0.001128,-0.001039,-0.000959,-0.000937,-0.000916,-0.000819,...,0.82978,3.5814,3.7667,1.1151,0.44572,0.20538,0.41882,1.4422,0.32451,0


In [7]:
import matplotlib.pyplot as plt
import seaborn as sns

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.metrics import r2_score

# Prepare data
X = df.drop('Label', axis=1)
y = df['Label']

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Data split into training and testing sets.")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

Data split into training and testing sets.
X_train shape: (184, 536)
X_test shape: (46, 536)


In [11]:
# Ridge Regression
ridge = Ridge(alpha=1.0) # You can tune the alpha parameter
ridge.fit(X_train, y_train)
y_pred_ridge = ridge.predict(X_test)
r2_ridge = r2_score(y_test, y_pred_ridge)

print(f"Ridge Regression R2 Score: {r2_ridge:.4f}")

Ridge Regression R2 Score: 0.7320


In [12]:
# Lasso Regression
lasso = Lasso(alpha=0.1) # You can tune the alpha parameter
lasso.fit(X_train, y_train)
y_pred_lasso = lasso.predict(X_test)
r2_lasso = r2_score(y_test, y_pred_lasso)

print(f"Lasso Regression R2 Score: {r2_lasso:.4f}")

Lasso Regression R2 Score: 0.3596


In [17]:
# ElasticNet Regression
elastic_net = ElasticNet(alpha=0.1, l1_ratio=0.00001) # Tune alpha and l1_ratio
elastic_net.fit(X_train, y_train)
y_pred_elastic_net = elastic_net.predict(X_test)
r2_elastic_net = r2_score(y_test, y_pred_elastic_net)

print(f"ElasticNet Regression R2 Score: {r2_elastic_net:.4f}")

ElasticNet Regression R2 Score: 0.6411


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Prepare data for plotting
plot_data_ridge = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred_ridge})
plot_data_lasso = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred_lasso})
plot_data_elastic_net = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred_elastic_net})

# --- Ridge Regression Plot ---
plt.figure(figsize=(8, 6))
sns.regplot(x='Actual', y='Predicted', data=plot_data_ridge, scatter_kws={'alpha':0.6}, line_kws={'color':'red'})
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', lw=2, label='Perfect Fit') # Perfect fit line
plt.title('Ridge Regression: Actual vs. Predicted Values')
plt.xlabel('Actual Values (y_test)')
plt.ylabel('Predicted Values (y_pred_ridge)')
plt.legend()
plt.grid(True)
plt.show()

# --- Lasso Regression Plot ---
plt.figure(figsize=(8, 6))
sns.regplot(x='Actual', y='Predicted', data=plot_data_lasso, scatter_kws={'alpha':0.6}, line_kws={'color':'red'})
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', lw=2, label='Perfect Fit') # Perfect fit line
plt.title('Lasso Regression: Actual vs. Predicted Values')
plt.xlabel('Actual Values (y_test)')
plt.ylabel('Predicted Values (y_pred_lasso)')
plt.legend()
plt.grid(True)
plt.show()

# --- ElasticNet Regression Plot ---
plt.figure(figsize=(8, 6))
sns.regplot(x='Actual', y='Predicted', data=plot_data_elastic_net, scatter_kws={'alpha':0.6}, line_kws={'color':'red'})
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', lw=2, label='Perfect Fit') # Perfect fit line
plt.title('ElasticNet Regression: Actual vs. Predicted Values')
plt.xlabel('Actual Values (y_test)')
plt.ylabel('Predicted Values (y_pred_elastic_net)')
plt.legend()
plt.grid(True)
plt.show()